In [ ]:
import pandas as pd
import supply_chain_utils as scu
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
import pulp

# Styling settings
plt.style.use('seaborn-v0_8-whitegrid')
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Generate synthetic historical daily demand dataset (1 year of data)
np.seed = 42
date_range = pd.date_range(start='2025-01-01', end='2025-12-31', freq='D')
n = len(date_range)

# Base trend + weekly seasonality + yearly seasonality + holiday spikes + noise
trend = np.linspace(50000, 65000, n)
weekly_seasonality = 8000 * np.sin(2 * np.pi * date_range.dayofweek / 7)
yearly_seasonality = 12000 * np.cos(2 * np.pi * date_range.dayofyear / 365)
noise = np.random.normal(0, 2500, n)

# External factor: Holiday/Peak travel periods (e.g., December and April holidays)
holidays_effect = np.where((date_range.month == 12) | (date_range.month == 4), 15000, 0)

demand = trend + weekly_seasonality + yearly_seasonality + holidays_effect + noise
demand = np.maximum(demand, 10000) # Floor at minimum operational throughput

df_history = pd.DataFrame({
    'ds': date_range,
    'y': demand,
    'is_peak_season': np.where(holidays_effect > 0, 1, 0)
})

# Visualize historical trends
plt.figure(figsize=(12, 5))
plt.plot(df_history['ds'], df_history['y'], color='#2E6BE6', lw=1.2)
plt.title('Historical Daily Product Demand (KPC Depot Throughput)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Volume (Litres)')
plt.show()

## Operational Assumptions & Parameters
* **Lead Time ($LT$):** 5 days for replenishment stock delivery from Mombasa/Nairobi terminal nodes to regional depots.
* **Target Service Level ($SL$):** $95\%$, corresponding to a Z-score ($Z$) of approximately $1.645$ to protect against stockouts during demand surges.
* **Holding Cost & Stockout Risk:** Stockouts incur severe commercial penalties and throughput stoppages, making safety stock optimization critical.

In [ ]:
# Initialize and fit Prophet model with external regressor
model = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
model.add_regressor('is_peak_season')

# Fit model on historical data
model.fit(df_history)

# Create future dataframe for 60 days horizon
future = model.make_future_dataframe(periods=60)

# Apply external regressor values for future dates
future['is_peak_season'] = np.where((future['ds'].dt.month == 12) | (future['ds'].dt.month == 4), 1, 0)

# Forecast
forecast = model.predict(future)

# Evaluate model performance using historical fitted values vs actuals
train_eval = forecast.iloc[:n]
mape = mean_absolute_percentage_error(df_history['y'], train_eval['yhat'])
rmse = root_mean_squared_error(df_history['y'], train_eval['yhat'])

print(f"Model Evaluation Metrics:\n- MAPE: {mape:.2%}\n- RMSE: {rmse:.2f} Litres")

# Plot forecast
model.plot(forecast)
plt.title('60-Day Demand Forecast with Prophet', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Extract forecast standard deviation of residuals (error variance)
forecast_residuals = df_history['y'] - train_eval['yhat']
std_dev_demand = np.std(forecast_residuals)

# Inventory parameters
lead_time_days = 5
service_level = 0.95
z_score = 1.645 # For 95% service level

# Calculations using formal equations
average_daily_demand = forecast['yhat'].tail(60).mean()
safety_stock = z_score * std_dev_demand * np.sqrt(lead_time_days)
reorder_point = (average_daily_demand * lead_time_days) + safety_stock

print(f"Inventory Optimization Results:")
print(f"- Average Daily Demand (Forecast): {average_daily_demand:,.2f} Litres")
print(f"- Demand Variability ($\sigma$): {std_dev_demand:,.2f} Litres")
print(f"- Calculated Safety Stock ($SS$): {safety_stock:,.2f} Litres")
print(f"- Calculated Reorder Point ($ROP$): {reorder_point:,.2f} Litres")

In [ ]:
# Simulate 60 days of inventory behavior comparing With vs Without Safety Stock
np.seed = 100
sim_days = 60
future_demand = forecast['yhat'].tail(sim_days).values + np.random.normal(0, std_dev_demand, sim_days)

initial_inventory = reorder_point + (average_daily_demand * lead_time_days)

inv_with_ss = initial_inventory
inv_without_ss = initial_inventory - safety_stock # Starts lower

stockouts_with = 0
stockouts_without = 0

history_sim = []

for day in range(sim_days):
    d_val = future_demand[day]
    
    # Policy With Safety Stock
    inv_with_ss -= d_val
    if inv_with_ss <= reorder_point:
        inv_with_ss += (average_daily_demand * lead_time_days) # Replenishment order arrived after LT
    if inv_with_ss < 0:
        stockouts_with += 1
        inv_with_ss = 0
        
    # Policy Without Safety Stock
    inv_without_ss -= d_val
    if inv_without_ss <= (reorder_point - safety_stock):
        inv_without_ss += (average_daily_demand * lead_time_days)
    if inv_without_ss < 0:
        stockouts_without += 1
        inv_without_ss = 0

print(f"Simulation Results over {sim_days} Days:")
print(f"- Stockout Events WITHOUT Safety Stock: {stockouts_without}")
print(f"- Stockout Events WITH Safety Stock: {stockouts_with}")

In [ ]:
# Formulate a distribution problem: Supplying 3 Depots from 2 Terminals to minimize transport cost
prob = pulp.LpProblem("KPC_Product_Distribution", pulp.LpMinimize)

terminals = ['Mombasa_Terminal', 'Nairobi_Terminal']
depots = ['Nakuru', 'Eldoret', 'Kisumu']

# Supply capacities
supply = {'Mombasa_Terminal': 1200000, 'Nairobi_Terminal': 900000}

# Depot demands (derived from forecast)
demand_depots = {'Nakuru': 500000, 'Eldoret': 700000, 'Kisumu': 600000}

# Transportation costs per unit volume
costs = {
    ('Mombasa_Terminal', 'Nakuru'): 15,
    ('Mombasa_Terminal', 'Eldoret'): 22,
    ('Mombasa_Terminal', 'Kisumu'): 28,
    ('Nairobi_Terminal', 'Nakuru'): 8,
    ('Nairobi_Terminal', 'Eldoret'): 14,
    ('Nairobi_Terminal', 'Kisumu'): 18
}

# Decision variables
route_vars = pulp.LpVariable.dicts("Route", ((t, d) for t in terminals for d in depots), lowBound=0, cat='Continuous')

# Objective Function
prob += pulp.lpSum(route_vars[t, d] * costs[t, d] for t in terminals for d in depots)

# Supply Constraints
for t in terminals:
    prob += pulp.lpSum(route_vars[t, d] for d in depots) <= supply[t], f"Supply_Constraint_{t}"

# Demand Constraints
for d in depots:
    prob += pulp.lpSum(route_vars[t, d] for t in terminals) >= demand_depots[d], f"Demand_Constraint_{d}"

# Solve
prob.solve()

print(f"Distribution Optimization Status: {pulp.LpStatus[prob.status]}")
print("Optimal Shipping Plan:")
for variable in prob.variables():
    if variable.varValue > 0:
        print(f"- {variable.name}: {variable.varValue:,.0f} Litres")
print(f"Total Minimized Transport Cost: KES {pulp.value(prob.objective):,.2f}")